In [1]:
import sys
import os
from jiwer import wer
import evaluate
from models.whisper_transcriber import WhisperTranscriber
from models.translator import ChunkTranslator
from utils.audio_preprocessing import preprocess_audio
from utils.audio_streaming import stream_audio


# Get the absolute path to the project root (parent directory of the current folder)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the project root to sys.path
sys.path.append(project_root)

c:\Users\aneas\anaconda3\envs\02466_AI_dubbing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\aneas\anaconda3\envs\02466_AI_dubbing\Lib\site-packages\ctranslate2\__init__.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### 1. Speech to text

In [11]:
transcriber = WhisperTranscriber(model_type="base.en")
translator = ChunkTranslator("Helsinki-NLP/opus-mt-en-da", context_length=2, num_beams=2)

data_path = "data/speaker_2.wav"
audio = preprocess_audio(data_path)

transcription_text = ""
translated_text = ""

for chunk in stream_audio(audio, frame_ms=200):
    transcription = transcriber.add_audio_chunk(chunk)
    if isinstance(transcription, str):
        transcription_text += " " + transcription
        translated = translator.translate_chunk(transcription)
        if isinstance(translated, str):
            translated_text += " " + translated


# At the end, flush any remaining audio
final_transcription = transcriber.flush()
transcription_text += " " + final_transcription
if isinstance(final_transcription, str):
    final_translation = translator.translate_chunk(final_transcription)
    if isinstance(final_translation, str):
        translated_text += " " + final_translation

translator.reset_context()

Loading Silero-VAD …
Silero-VAD initialised!
Loading Faster Whisper model: base.en …
Whisper model loaded!


c:\Users\aneas\anaconda3\envs\02466_AI_dubbing\Lib\site-packages\transformers\models\marian\tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


4.0
4.036
4.004
4.004
4.228
4.036
4.004
4.004
4.004
4.516
4.004
4.42
4.004
4.004
4.708
4.004
4.004
4.004
4.004
4.004
4.004
4.004
4.004
4.356
4.004
4.004
4.004
4.004
4.004
4.004


In [12]:
def clean_text(text):
    text = text.replace("'", "")
    text = text.replace(".", "")
    text = text.replace(",", "")
    text = text.replace("-", " ")
    text = text.lower()
    return text

transcription_text = clean_text(transcription_text)
translated_text = clean_text(translated_text)

In [13]:
with open("data/speaker_2.txt", "r", encoding="utf-8") as f:
    ref_transcription = f.read()
    
wer_score = wer(reference=ref_transcription, hypothesis=transcription_text)
wer_score

0.0726643598615917

### Text translation

In [15]:
with open("data/speaker_2.txt", "r", encoding="utf-8") as f:
    ref_translation = f.read()

In [ ]:
comet = evaluate.load("comet")
comet_score = comet.compute(
    predictions=[translated_text],
    references=[ref_translation],
    sources=[transcription_text],
)

comet_score

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 3545.48it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.1.post0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\aneas\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`
Encoder model frozen.
c:\Users\aneas\anaconda3\envs\02466_AI_dubbing\Lib\site-packages\pytorch_lightning\core\saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


{'mean_score': 0.992570161819458, 'scores': [0.992570161819458]}

In [ ]:
# translated_text, ref_translation, transcription_text